In [1]:
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
print("cwd:", os.getcwd())

import polars as pl

cwd: /home/johnh/projects/microstructure-benchmark


In [2]:
n = 1_000_000

df = pl.select(
    (pl.datetime(2025, 9, 2, 13, 30, 0) + pl.duration(milliseconds=pl.arange(0, n, eager=True))).alias("timestamp"),
    ((pl.arange(0, n, eager=True) % 1000).cast(pl.Float64) + 100.0).alias("price"),
    ((pl.arange(0, n, eager=True) % 50 + 1).cast(pl.Int64)).alias("size"),
)

df.head(), df.shape

(shape: (5, 3)
 ┌─────────────────────────┬───────┬──────┐
 │ timestamp               ┆ price ┆ size │
 │ ---                     ┆ ---   ┆ ---  │
 │ datetime[μs]            ┆ f64   ┆ i64  │
 ╞═════════════════════════╪═══════╪══════╡
 │ 2025-09-02 13:30:00     ┆ 100.0 ┆ 1    │
 │ 2025-09-02 13:30:00.001 ┆ 101.0 ┆ 2    │
 │ 2025-09-02 13:30:00.002 ┆ 102.0 ┆ 3    │
 │ 2025-09-02 13:30:00.003 ┆ 103.0 ┆ 4    │
 │ 2025-09-02 13:30:00.004 ┆ 104.0 ┆ 5    │
 └─────────────────────────┴───────┴──────┘,
 (1000000, 3))

In [3]:
out = Path("data/parquet/demo.parquet")
out.parent.mkdir(parents=True, exist_ok=True)

df.write_parquet(out, compression="zstd")
out.stat().st_size / 1e6

1.873375

In [4]:
result = (
    pl.scan_parquet("data/parquet/demo.parquet")
    .filter(pl.col("price") > 0)
    .group_by_dynamic("timestamp", every="1s")
    .agg(pl.col("size").sum().alias("size_sum"))
    .collect()
)

result.head(10), result.shape

(shape: (10, 2)
 ┌─────────────────────┬──────────┐
 │ timestamp           ┆ size_sum │
 │ ---                 ┆ ---      │
 │ datetime[μs]        ┆ i64      │
 ╞═════════════════════╪══════════╡
 │ 2025-09-02 13:30:00 ┆ 25500    │
 │ 2025-09-02 13:30:01 ┆ 25500    │
 │ 2025-09-02 13:30:02 ┆ 25500    │
 │ 2025-09-02 13:30:03 ┆ 25500    │
 │ 2025-09-02 13:30:04 ┆ 25500    │
 │ 2025-09-02 13:30:05 ┆ 25500    │
 │ 2025-09-02 13:30:06 ┆ 25500    │
 │ 2025-09-02 13:30:07 ┆ 25500    │
 │ 2025-09-02 13:30:08 ┆ 25500    │
 │ 2025-09-02 13:30:09 ┆ 25500    │
 └─────────────────────┴──────────┘,
 (1000, 2))

In [5]:
import time

t0 = time.perf_counter()
(
    pl.scan_parquet("data/parquet/demo.parquet")
    .filter(pl.col("price") > 0)
    .group_by_dynamic("timestamp", every="1s")
    .agg(pl.col("size").sum())
    .collect()
)
print(f"{time.perf_counter() - t0:.3f}s")

0.040s


In [6]:
pl.read_parquet("data/parquet/mnq_mbp1_2025-09-02_smoke.parquet").head(20)

ts_recv,ts_event,rtype,publisher_id,instrument_id,action,side,depth,price,size,flags,ts_in_delta,sequence,bid_px_00,ask_px_00,bid_sz_00,ask_sz_00,bid_ct_00,ask_ct_00,symbol
"datetime[ns, UTC]","datetime[ns, UTC]",u8,u16,u32,str,str,u8,f64,u32,u8,i32,u32,f64,f64,u32,u32,u32,u32,str
2025-09-02 13:30:00.000082355 UTC,2025-09-02 13:29:59.999824367 UTC,1,1,42003472,"""A""","""A""",0,23100.75,1,128,11563,28593008,23100.25,23100.75,6,4,5,4,"""MNQ.c.0"""
2025-09-02 13:30:00.000133670 UTC,2025-09-02 13:29:59.999854931 UTC,1,1,42003472,"""C""","""B""",0,23100.25,1,128,11867,28593013,23100.25,23100.75,5,4,4,4,"""MNQ.c.0"""
2025-09-02 13:30:00.000593777 UTC,2025-09-02 13:29:59.999908843 UTC,1,1,42003472,"""A""","""A""",0,23100.75,1,128,15218,28593014,23100.25,23100.75,5,5,4,5,"""MNQ.c.0"""
2025-09-02 13:30:00.000593777 UTC,2025-09-02 13:29:59.999945821 UTC,1,1,42003472,"""A""","""A""",0,23100.75,1,128,15218,28593014,23100.25,23100.75,5,6,4,6,"""MNQ.c.0"""
2025-09-02 13:30:00.000593777 UTC,2025-09-02 13:29:59.999947971 UTC,1,1,42003472,"""A""","""A""",0,23100.75,1,128,15218,28593014,23100.25,23100.75,5,7,4,7,"""MNQ.c.0"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-09-02 13:30:00.003612551 UTC,2025-09-02 13:30:00.001161211 UTC,1,1,42003472,"""A""","""A""",0,23100.5,1,128,11602,28593045,23100.0,23100.5,5,4,4,4,"""MNQ.c.0"""
2025-09-02 13:30:00.003633953 UTC,2025-09-02 13:30:00.001170647 UTC,1,1,42003472,"""C""","""B""",0,23100.0,2,128,11830,28593047,23100.0,23100.5,3,4,3,4,"""MNQ.c.0"""
2025-09-02 13:30:00.003746997 UTC,2025-09-02 13:30:00.001262803 UTC,1,1,42003472,"""A""","""A""",0,23100.5,1,128,11390,28593059,23100.0,23100.5,3,5,3,5,"""MNQ.c.0"""
